# STEP 1: Honda Data Loading Playground (UPDATED)

**Purpose:** Load and explore Honda data from correct data source

**Data Source:** `accy_v2/data/landing_zone/honda/may_2026/CVIC5D.xlsx`

**Updated:** Sep 28, 2026

Run each cell to load and verify data.

## Setup: Import Libraries

In [8]:
import pandas as pd
import os
from pathlib import Path
import sys
import json
import re
from datetime import datetime

print("Libraries loaded successfully")

Libraries loaded successfully


## STEP 1: Load from Correct Data Source (may_2026)

In [11]:
# STEP 1: Load from Correct Data Source (may_2026)
# Using absolute path - works from any notebook location

civic_5d_file = Path("C:/Users/paxm/OneDrive - PBS SYSTEMS/Desktop/Office/Projects/OEM Accessory project/OEM_Accessories_v1/accy_v2/data/landing_zone/honda/may_2026/CVIC5D.xlsx")

print(f"Loading from: {civic_5d_file}\n")

if not civic_5d_file.exists():
    print(f"[ERROR] File not found: {civic_5d_file}")
    exit(1)

print(f"[OK] File found: {civic_5d_file.name}")
print(f"  Size: {civic_5d_file.stat().st_size / (1024**2):.2f} MB\n")

# Set landing_zone for reference
landing_zone = civic_5d_file.parent

# List all files
excel_files = list(landing_zone.glob("*.xlsx*"))
print(f"Total Honda files in may_2026: {len(excel_files)}\n")

for i, file in enumerate(excel_files, 1):
    size_mb = file.stat().st_size / (1024**2)
    print(f"{i:2d}. {file.name:<50} ({size_mb:.2f} MB)")

Loading from: C:\Users\paxm\OneDrive - PBS SYSTEMS\Desktop\Office\Projects\OEM Accessory project\OEM_Accessories_v1\accy_v2\data\landing_zone\honda\may_2026\CVIC5D.xlsx

[OK] File found: CVIC5D.xlsx
  Size: 5.17 MB

Total Honda files in may_2026: 1

 1. CVIC5D.xlsx                                        (5.17 MB)


## STEP 2: Identify Sheets in File

In [12]:
xl = pd.ExcelFile(civic_5d_file)
sheet_names = xl.sheet_names

print(f"Sheets in file ({len(sheet_names)} total):\n")
for i, sheet in enumerate(sheet_names, 1):
    print(f"  {i}. {sheet}")

# Find EN and FR sheets
en_sheets = [s for s in sheet_names if "_APP_EN" in s]
fr_sheets = [s for s in sheet_names if "_APP_FR" in s]
select_province = [s for s in sheet_names if "SELECT" in s.upper()]

print(f"\nEN Sheets: {en_sheets}")
print(f"FR Sheets: {fr_sheets}")
print(f"Select Province: {select_province}")

Sheets in file (6 total):

  1. 26MY_CIV5D_DATA
  2. SELECT PROVINCE
  3. 26MY_CIV5D_APP_EN
  4. 26MY_CIV5D_APP_FR
  5. 26MY_CIV5D_SELL-EN
  6. 26MY_CIV5D_SELL-FR

EN Sheets: ['26MY_CIV5D_APP_EN']
FR Sheets: ['26MY_CIV5D_APP_FR']
Select Province: ['SELECT PROVINCE']


## STEP 3: Load EN Sheet Data

In [17]:
en_sheet = en_sheets[0] if en_sheets else None

if en_sheet:
    df_civic = pd.read_excel(civic_5d_file, sheet_name=en_sheet)
    
    print(f"Sheet: {en_sheet}")
    print(f"Data shape: {len(df_civic)} rows x {len(df_civic.columns)} columns\n")
    
    print(f"Column names ({len(df_civic.columns)} total):")
    for i, col in enumerate(df_civic.columns, 1):
        print(f"  {i:2d}. {col}")
else:
    print("[ERROR] No EN sheet found")
    df_civic = None

Sheet: 26MY_CIV5D_APP_EN
Data shape: 170 rows x 17 columns

Column names (17 total):
   1. Unnamed: 0
   2. Unnamed: 1
   3. Unnamed: 2
   4. Unnamed: 3
   5. Unnamed: 4
   6. Unnamed: 5
   7. Unnamed: 6
   8. Unnamed: 7
   9. Unnamed: 8
  10. Unnamed: 9
  11. Unnamed: 10
  12. Unnamed: 11
  13. Unnamed: 12
  14. Unnamed: 13
  15. Unnamed: 14
  16. Unnamed: 15
  17. Unnamed: 16


## STEP 4: Verify Data Integrity

In [18]:
if df_civic is not None:
    # Check for Racing Package
    racing_found = df_civic.astype(str).apply(
        lambda row: any("Racing" in str(val) for val in row.values),
        axis=1
    ).any()
    
    if racing_found:
        print("[OK] Racing Package - Type R found in data\n")
        
        # Find and display the row
        for idx, row in df_civic.iterrows():
            row_text = " ".join(str(val) for val in row.values if pd.notna(val))
            if "Racing" in row_text and "TYPER" in row_text:
                print(f"  Location: Row {idx}")
                print(f"  Content: {row_text[:150]}\n")
                break
    else:
        print("[ERROR] Racing Package not found - data may be incomplete\n")
    
    # Check data completeness
    print(f"Data completeness check:")
    print(f"  Total rows: {len(df_civic)}")
    print(f"  Total columns: {len(df_civic.columns)}")
    print(f"  Null values: {df_civic.isnull().sum().sum()}")
    print(f"\n[OK] Data loaded successfully from may_2026 landing zone")
else:
    print("[ERROR] Could not verify data")

[OK] Racing Package - Type R found in data

  Location: Row 12
  Content: 26TYPER-RACE Racing Package - Type R 26TYPER-RACE 3928 6431 0.9999999999999999 6612 0 0.0 0.0 0.0 •

Data completeness check:
  Total rows: 170
  Total columns: 17
  Null values: 1415

[OK] Data loaded successfully from may_2026 landing zone


## STEP 5: Extract Section Boundaries

In [21]:
# Helper function to check if a row is empty
def is_row_empty(row):
    """Check if a row is completely empty (all NaN or all whitespace)."""
    for val in row.values:
        if pd.notna(val):
            val_str = str(val).strip()
            if val_str and val_str != 'nan':
                return False
    return True

# Identify section boundaries using section markers
def identify_section_boundaries(df):
    """
    Find section boundaries in the data.
    Auto-detects sections by looking for section markers (e.g., "1.0", "2.0", etc.)
    """
    df_str = df.astype(str)
    section_starts = []
    
    # Find all section markers (pattern: digit(s).digit(s) followed by text)
    section_pattern = r'^\d+\.\d+\s+(.+)$'
    
    for idx, row in df_str.iterrows():
        for cell_value in row.values:
            cell_str = str(cell_value).strip()
            match = re.match(section_pattern, cell_str)
            if match:
                section_name = match.group(1).strip()
                section_starts.append((idx, section_name, cell_str))
                break
    
    # Build section boundaries
    sections = {}
    for i, (marker_row, section_name, marker_text) in enumerate(section_starts):
        # Data starts AFTER the marker row
        data_start = marker_row + 1
        
        # Data ends BEFORE the next section marker (or at end of dataframe)
        if i + 1 < len(section_starts):
            next_marker_row = section_starts[i + 1][0]
            data_end = next_marker_row - 1
        else:
            # Last section: goes to end of dataframe
            data_end = len(df) - 1
        
        # Count rows in this section
        if data_start <= data_end:
            total_rows = data_end - data_start + 1
            
            # Count empty vs non-empty rows
            empty_rows = 0
            non_empty_rows = 0
            for row_idx in range(data_start, data_end + 1):
                if is_row_empty(df.iloc[row_idx]):
                    empty_rows += 1
                else:
                    non_empty_rows += 1
        else:
            total_rows = 0
            empty_rows = 0
            non_empty_rows = 0
        
        sections[section_name] = {
            "marker_row": marker_row,
            "data_start": data_start,
            "data_end": data_end,
            "total_rows": total_rows,
            "empty_rows": empty_rows,
            "non_empty_rows": non_empty_rows
        }
    
    return sections

# Extract sections
if df_civic is not None:
    print(f"Analyzing {len(df_civic)} rows...\n")
    
    sections = identify_section_boundaries(df_civic)
    
    if sections:
        print(f"[OK] Found {len(sections)} sections:\n")
        
        total_all_rows = 0
        for section_name, bounds in sections.items():
            print(f"  {section_name}")
            print(f"    Marker row: {bounds['marker_row']}")
            print(f"    Data rows: {bounds['data_start']}-{bounds['data_end']}")
            print(f"    Total rows in section: {bounds['total_rows']}")
            total_all_rows += bounds['total_rows']
        
        print(f"\n  Total rows (all sections): {total_all_rows}")
    else:
        print("[ERROR] No sections found")
else:
    print("[ERROR] df_civic not loaded")

Analyzing 170 rows...

[OK] Found 6 sections:

  Packages and Kits
    Marker row: 9
    Data rows: 10-55
    Total rows in section: 46
  Wheels and Wheel Accessories
    Marker row: 56
    Data rows: 57-71
    Total rows in section: 15
  Exterior
    Marker row: 72
    Data rows: 73-122
    Total rows in section: 50
  Interior
    Marker row: 123
    Data rows: 124-142
    Total rows in section: 19
  Electrical
    Marker row: 143
    Data rows: 144-152
    Total rows in section: 9
  Paint Pens
    Marker row: 153
    Data rows: 154-169
    Total rows in section: 16

  Total rows (all sections): 155


## STEP 6: Section Display Function

In [26]:
def display_section(section_ref, sections, df):
    """
    Display all rows for a given section with debugging info.
    
    Args:
        section_ref: Either a section name (string) or section index (integer, 0-based)
        sections: Dictionary of sections from identify_section_boundaries()
        df: DataFrame containing the data
    
    Examples:
        display_section("Packages and Kits", sections, df_civic)  # By name
        display_section(0, sections, df_civic)                    # By index (first section)
        display_section(2, sections, df_civic)                    # By index (third section)
    """
    
    # Convert section reference to section name
    section_names = list(sections.keys())
    
    if isinstance(section_ref, int):
        # Handle index-based access
        if section_ref < 0 or section_ref >= len(section_names):
            print(f"[ERROR] Section index {section_ref} out of range (0-{len(section_names)-1})")
            print(f"\nAvailable sections:")
            for i, name in enumerate(section_names):
                bounds = sections[name]
                print(f"  [{i}] {name} (rows {bounds['data_start']}-{bounds['data_end']}, {bounds['non_empty_rows']} data rows)")
            return None
        section_name = section_names[section_ref]
    else:
        # Handle name-based access
        section_name = section_ref
        if section_name not in sections:
            print(f"[ERROR] Section '{section_name}' not found")
            print(f"\nAvailable sections:")
            for i, name in enumerate(section_names):
                bounds = sections[name]
                print(f"  [{i}] {name} (rows {bounds['data_start']}-{bounds['data_end']}, {bounds['non_empty_rows']} data rows)")
            return None
    
    bounds = sections[section_name]
    data_start = bounds['data_start']
    data_end = bounds['data_end']
    
    print("="*100)
    print(f"SECTION: {section_name}")
    print("="*100)
    print(f"\nMarker row: {bounds['marker_row']}")
    print(f"Data range: rows {data_start} to {data_end}")
    print(f"Total rows: {bounds['total_rows']} (Empty: {bounds['empty_rows']}, Data: {bounds['non_empty_rows']})")
    print(f"\n")
    
    # Show all data rows
    section_data_all = df.iloc[data_start:data_end+1]
    
    with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', None):
        display(section_data_all)
    
    print("\n" + "="*100 + "\n")
    
    return section_data_all

print("[OK] display_section() function ready")
print("\nUsage examples:")
print("  display_section(0, sections, df_civic)  # Display first section by index")
print("  display_section(1, sections, df_civic)  # Display second section by index")
print("  display_section('Packages and Kits', sections, df_civic)  # Display by name")

[OK] display_section() function ready

Usage examples:
  display_section(0, sections, df_civic)  # Display first section by index
  display_section(1, sections, df_civic)  # Display second section by index
  display_section('Packages and Kits', sections, df_civic)  # Display by name


## STEP 7: Display Section Example

In [ ]:

# Display the first section (Packages and Kits)
packages_section = display_section(5, sections, df_civic)
 

SECTION: Paint Pens

Marker row: 153
Data range: rows 154 to 169
Total rows: 16 (Empty: 2, Data: 14)




,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16
154,NaN,NaN,Description,Part Number,Dealer Net,List Price,FRT,Installed Price,Application,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Comments
155,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0.0,0.0,SPORT,SPORT \nHybrid,0.0,Sport Touring Hybrid,Type R,NaN
156,NaN,TUCH-NH883P-A1,Platinum White Pearl (NH-883P),TUCH-NH883P-A1,13.25,22,0,22,0,0.0,0.0,•,•,0.0,•,NaN,NaN
157,NaN,TUCH-NH0-A1,Championship White (NH-0),TUCH-NH0-A1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,•,NaN
158,NaN,TUCH-NH932M-A1,Solar Reflection Silver (NH-932M),TUCH-NH932M-A1,NaN,NaN,NaN,NaN,0,0.0,0.0,•,NaN,0.0,NaN,NaN,NaN
159,NaN,TUCH-NH904M-A1,Meteoroid Gray Pearl (NH-904M),TUCH-NH904M-A1,NaN,NaN,NaN,NaN,0,0.0,0.0,NaN,•,0.0,NaN,NaN,NaN
160,NaN,TUCH-NH731P-A1,Crystal Black Pearl (NH-731P),TUCH-NH731P-A1,NaN,NaN,NaN,NaN,0,0.0,0.0,•,•,0.0,•,NaN,NaN
161,NaN,TUCH-R513-A1,Rallye Red (R-513),TUCH-R513-A1,NaN,NaN,NaN,NaN,0,0.0,0.0,NaN,•,0.0,•,•,NaN
162,NaN,TUCH-B637P-A1,Boost Blue Pearl (B-637P),TUCH-B637P-A1,NaN,NaN,NaN,NaN,0,0.0,0.0,NaN,NaN,0.0,•,•,NaN
163,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Summary: Data Loading & Section Extraction Complete

**Status: Ready for Next Phase**
- ✓ Data loaded from `may_2026/CVIC5D.xlsx`
- ✓ 6 sections identified and extracted
- ✓ `display_section()` function ready for exploration
- ✓ All sections accessible by index (0-5) or name

**Try these commands:**
```python
# View different sections
display_section(1, sections, df_civic)  # Wheels
display_section(2, sections, df_civic)  # Interior
display_section(3, sections, df_civic)  # Exterior
display_section(4, sections, df_civic)  # Cargo
display_section(5, sections, df_civic)  # General
```

**Next Step:** Flatten the data for analysis